In [1]:
!pip install transformers[torch] datasets scikit-learn pandas

import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import io
import torch

Aplicam lemmatizarea pentru a permite modelului sa invete si sa se adapteze mai rapid si eliminam semnele de punctuatie fiindca nu ajuta la invatarea modelului


In [2]:
import spacy

# Descarcă și încarcă modelul pentru limba română
!python -m spacy download ro_core_news_sm
nlp = spacy.load("ro_core_news_sm")

def lemmatize_text(text):
    doc = nlp(text.lower())
    # Extragem forma de bază (lemma) și eliminăm semnele de punctuație
    lemmas = [token.lemma_ for token in doc if not token.is_punct]
    return " ".join(lemmas)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 68.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ro_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
data = []
with open('prompts.csv', 'r', encoding='utf-8') as f:
    for line in f:
        # Împărțim linia doar după ULTIMA virgulă
        if ',' in line:
            parts = line.strip().rsplit(',', 1)
            data.append(parts)

df = pd.DataFrame(data, columns=['text', 'label'])

In [4]:
#ELIMINAREA VIRGULELOR INTERNE
def preprocess_for_bert(text):
    # Eliminăm virgulele interne pentru a simplifica vectorizarea
    text = text.replace(',', ' ')
    # Eliminăm spațiile multiple create
    text = " ".join(text.split())
    return text

df['text'] = df['text'].apply(preprocess_for_bert)

In [5]:
print("Se procesează lemmatizarea... poate dura un minut.")
df['text_lemmatizat'] = df['text'].apply(lemmatize_text)

print(df[['text', 'text_lemmatizat']].head())

Se procesează lemmatizarea... poate dura un minut.
                                                text  \
0  frate iar mi-a ajuns comanda stricată… bătaie ...   
1         nu recomand deloc suportu clienti = ZERO 😒   
2  mi ati luat banii si produsul nici acum nu a v...   
3      cea mai proasta experienta ever cu firma asta   
4    am stat 40 min in asteptare si nimic… penibil 😂   

                                     text_lemmatizat  
0  frate iar eu avea ajunge comandă stricat bătai...  
1        nu recomanda deloc suportu clienti = zero 😒  
2  eu ati lua ban si produs nici acum nu avea veni 👍  
3     cel mai proasta experient ever cu firmă acesta  
4     avea sta 40 min in asteptar si nimic penibil 😂  


Am folosit modelul pre-antrenat Romanian BERT pentru tokenizare, dezvoltat de Institutul de Cercetare pentru Inteligență Artificială "Mihai Drăgănescu. Datorită acestui model, nu am eliminat stopwords.


In [7]:
label_map = {
    "RECLAMATIE": 0,
    "MULTUMIRE": 1,
    "SALUT": 2,
    "SMALL_TALK": 3,
    "INTREBARE": 4,
    "PREZENTARE": 5,
    "HELP": 6
}
df['label_num'] = df['label'].map(label_map)

df['label_num'] = df['label_num'].astype(int)

# 2. Împărțim datele (80% antrenare, 20% validare)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text_lemmatizat'].tolist(),
    df['label_num'].tolist(),
    test_size=0.2,
    random_state=42
)

# 3. Inițializăm Tokenizer-ul pentru Română
model_name = "dumitrescustefan/bert-base-romanian-cased-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 4. Funcția de tokenizare (transformă textul în ID-uri numerice)
def tokenize_data(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")

train_encodings = tokenize_data(train_texts)
val_encodings = tokenize_data(val_texts)

# 5. Creăm clasa de Dataset pentru PyTorch
class ChatbotDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ChatbotDataset(train_encodings, train_labels)
val_dataset = ChatbotDataset(val_encodings, val_labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Antrenarea efectiva a modelului

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
    report_to="none"                  # Dezactivează raportarea externă pentru a evita alte erori
)

# Cream obiectul Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# START ANTRENĂRII
print("Antrenarea a început... ar trebui să dureze câteva minute.")
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Antrenarea a început... ar trebui să dureze câteva minute.


Epoch,Training Loss,Validation Loss
1,No log,0.728401
2,No log,0.301482
3,No log,0.213264
4,No log,0.224572
5,No log,0.221789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=320, training_loss=0.43776650428771974, metrics={'train_runtime': 205.3323, 'train_samples_per_second': 12.37, 'train_steps_per_second': 1.558, 'total_flos': 19580041485000.0, 'train_loss': 0.43776650428771974, 'epoch': 5.0})

In [9]:
model.save_pretrained("./intent_model_final")
tokenizer.save_pretrained("./intent_model_final")
print("Modelul a fost salvat și este gata pentru livrare!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelul a fost salvat și este gata pentru livrare!
